# Task 1.1 — Core Contribution / Architecture

**Paper:** Nyström Method vs Random Fourier Features: A Theoretical and Empirical Comparison  
**Authors:** Tianbao Yang, Yu-Feng Li, Mehrdad Mahdavi, Rong Jin, Zhi-Hua Zhou  
**Venue:** NIPS 2012

---

## Step-by-Step Method Description

Both methods approximate a kernel function k(x, y) using an explicit low-dimensional feature map ψ(·) such that k(x, y) ≈ ⟨ψ(x), ψ(y)⟩, enabling fast linear SVM training.

---

### Shared Pre-step: Kernel Selection

- **Description:** A shift-invariant kernel k(x, y) = k(x − y) is chosen (e.g., RBF: k(x,y) = exp(−γ‖x−y‖²)). Both methods require a kernel that can be approximated by a finite-dimensional feature map.
- **Reference:** Section 2 (Preliminaries); the paper works with shift-invariant kernels throughout.
- **Purpose:** Establishes the class of kernels for which both approximations are theoretically valid.

---

### Method A: Random Fourier Features (RFF)

#### Step 1: Sample Random Frequencies

- **Description:** D frequency vectors ω₁, …, ω_D are sampled i.i.d. from the spectral distribution p(ω) of the kernel, which is the Fourier transform of k. For the RBF kernel, p(ω) = N(0, 2γI).
- **Reference:** Equation (2), Section 2.1 (based on Rahimi & Recht 2007, cited in the paper).
- **Purpose:** Bochner's theorem guarantees that k(x,y) = E_ω[cos(ωᵀx + b)cos(ωᵀy + b)], so random samples from p(ω) provide an unbiased estimator.

#### Step 2: Construct Explicit Feature Map

- **Description:** Each training/test point x is mapped to a D-dimensional vector: z(x) = √(2/D) [cos(ω₁ᵀx + b₁), …, cos(ω_Dᵀx + b_D)] where bᵢ ~ Uniform[0, 2π].
- **Reference:** Equation (3), Section 2.1.
- **Purpose:** Converts the kernel computation into a dot product: k(x,y) ≈ z(x)ᵀz(y). Now a linear SVM in this feature space approximates a kernel SVM.

---

### Method B: Nyström Method

#### Step 1: Sample Landmark Points

- **Description:** m landmark points {x̃₁, …, x̃_m} are sampled (uniformly or via k-means) from the training set. These serve as basis vectors for the approximation.
- **Reference:** Section 2.2, Equation (5).
- **Purpose:** Data-adaptive sampling means landmarks capture the actual structure of the training distribution, unlike RFF's data-independent random projections.

#### Step 2: Compute Kernel Sub-matrices

- **Description:** Two sub-matrices are formed: K_{mn} (m×n kernel matrix between landmarks and all training points) and K_{mm} (m×m kernel matrix among landmarks).
- **Reference:** Equation (5), Section 2.2.
- **Purpose:** These sub-matrices encode all pairwise kernel relationships involving the landmark points.

#### Step 3: Eigendecomposition of K_{mm}

- **Description:** K_{mm} = U Λ Uᵀ is computed, and the Nyström feature map is: φ(x) = Λ^{-1/2} Uᵀ k_m(x), where k_m(x) = [k(x̃₁,x), …, k(x̃_m,x)]ᵀ.
- **Reference:** Equation (6), Section 2.2.
- **Purpose:** Produces a rank-m approximation of the full n×n kernel matrix. The approximation quality is determined by how well the m eigenvalues of K_{mm} capture the spectral content of the full kernel matrix.

---

### Shared Final Step: Train Linear SVM

- **Description:** The approximate feature maps ψ(x) (from either RFF or Nyström) are fed into a linear SVM, which solves: min_{w,b} ½‖w‖² + C·Σ max(0, 1 − yᵢ(wᵀψ(xᵢ)+b)).
- **Reference:** Section 2 (Introduction to both methods); the SVM training objective is standard.
- **Purpose:** Linear SVM on approximate features scales as O(n·D) rather than O(n²) for a kernel SVM, making the pipeline tractable for large datasets.

---

### Key Theoretical Finding (Paper's Core Contribution)

- **Description:** The paper proves (Theorem 1 vs Theorem 2) that Nyström achieves a tighter generalization bound than RFF when the eigenvalues of the kernel matrix decay rapidly. Specifically, Nyström's error depends on the tail of the eigenspectrum (Σ_{i>m} λᵢ), while RFF's error depends on D^{-1/2} regardless of spectral structure.
- **Reference:** Theorems 1 and 2, Section 3.
- **Purpose:** This is the paper's primary contribution — explaining *why* Nyström outperforms RFF on structured data, not just showing empirically that it does.

---

## Final Summary Sentence

This paper addresses the problem of scalable kernel SVM training by providing the first rigorous theoretical comparison between Nyström and Random Fourier Features kernel approximations, and the authors claim their analysis proves that Nyström achieves superior generalization bounds whenever the kernel matrix has fast-decaying eigenvalues — a property that holds for many real-world structured datasets.
